# 11장 실습 — 시뮬레이션 루프 직접 짜기 (채점)

채울 파일은 `labs/ch11_simloop.py`입니다.

수요, 소요시간 모형, 배차 방법을 1분 단위 루프에 연결합니다.
3장의 최단경로가 소요시간을 주고, 8장의 수요가 호출을 만들고, 10장의 배차가 차를 고릅니다.
이 장에서 만드는 것은 그것들을 1분마다 부르는 바깥 루프입니다.

채점기는 기록 형식과 상태 불변식을 검사하고, 평균 대기시간을 DTUMOS 녹화본과 비교합니다. 평균 대기시간 차이의 허용 범위는 1.5분입니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

import ch11_simloop as sol      # 여러분이 채우는 파일

## 1. 분 단위 처리 순서 (교재 11.1)

1분마다 이 다섯 가지를 순서대로 합니다.

1. 이 분에 들어온 호출을 대기열에 넣습니다
2. 너무 오래 기다린 승객을 실패로 처리합니다
3. 대기 승객과 빈 차로 비용행렬을 만들어 배차합니다
4. 배차된 차의 도착 시각을 정합니다
5. 이 분의 상태를 한 줄로 기록합니다

접수보다 배차를 먼저 실행하면 해당 분에 들어온 호출은 다음 분의 배차 대상이 됩니다.

In [ ]:
from smartmob.data import load_demand, load_vehicles

demand = load_demand("hanam")
vehicles = load_vehicles("hanam")

print(f"호출 {len(demand):,}건, 차량 {len(vehicles)}대")
print(f"차량 컬럼: {list(vehicles.columns)}")
vehicles.head(3)

차량마다 `work_start` 와 `work_end` 가 있습니다.
각 분의 근무 차량 수는 `work_start <= minute < work_end`를 만족하는 차량 수입니다.

## 2. 1시간 실행

먼저 18~19시 한 시간의 기록 길이와 컬럼을 확인한 뒤 전체 구간을 실행합니다.

In [ ]:
banner("18:00 ~ 19:00 (60분)")
try:
    small = sol.simulate(demand, vehicles, 1080, 1140)
    print(small.record.head())
    print()
    print(small.summary())
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 3. 전체 구간

In [ ]:
import time

banner("18:00 ~ 24:00 (360분)")
try:
    t0 = time.perf_counter()
    run = sol.simulate(demand, vehicles, 1080, 1440)
    print(f"{time.perf_counter() - t0:.1f}초")

    expect("record 행 수", len(run.record), 360)
    expect("record 컬럼", list(run.record.columns),
           ["time", "waiting_passenger_cnt", "fail_passenger_cnt",
            "empty_vehicle_cnt", "driving_vehicle_cnt"])

    s = run.summary()
    print(f"    서비스율 {s['service_rate']:.1%}, 평균대기 {s['avg_waiting_time_min']:.2f}분")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 4. DTUMOS 녹화본과 비교 (교재 11.5)

같은 수요와 같은 차량으로 DTUMOS 를 돌린 결과가 저장소에 녹화되어 있습니다.
두 결과를 나란히 놓습니다.

In [ ]:
from smartmob import Dtumos

engine = Dtumos().run_simulation(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)
engine.summary()

In [ ]:
import pandas as pd

from smartmob.teaching.metrics import kpi_table

try:
    run = sol.simulate(demand, vehicles, 1080, 1440)
    table = pd.DataFrame({
        "내 루프": kpi_table(run),
        "DTUMOS": kpi_table(engine),
    })
    display(table.round(3))
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

로컬 루프는 요청 1,000건과 직선거리 소요시간 모형을 사용하고, 녹화본 결과에는 요청 990건이 들어 있습니다. 따라서 이 비교는 동일 입력에 대한 구현 검증이 아니라 채점용 회귀 기준입니다.

## 5. 채점

In [ ]:
from check import check

try:
    report = check("ch11")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 6. 차량 대수별 결과 (교재 11.7)

차량 대수를 바꿔 서비스율, 평균 대기시간, 가동률을 비교합니다. 아래 셀은 교재 구현을 사용하며, 구현을 마친 뒤 `sol.simulate`로 바꿔 실행할 수 있습니다.

In [ ]:
from smartmob.teaching.simloop import simulate as reference

rows = []
for n_vehicles in [20, 40, 60, 80, 120, 160]:
    r = reference(demand, vehicles.head(n_vehicles), 1080, 1440)
    s = r.summary()
    rows.append({
        "차량": n_vehicles,
        "서비스율": round(s["service_rate"], 3),
        "평균대기_분": round(s["avg_waiting_time_min"], 2),
        "최대대기_분": round(s["max_waiting_time_min"], 1),
        "가동률": round(s["utilization"], 3),
    })

sweep = pd.DataFrame(rows)
sweep

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sweep["차량"], sweep["평균대기_분"], marker="o", color="#4C6EF5", label="평균 대기 (분)")
ax1.set_xlabel("차량 대수")
ax1.set_ylabel("평균 대기 (분)", color="#4C6EF5")

ax2 = ax1.twinx()
ax2.plot(sweep["차량"], sweep["가동률"], marker="s", color="crimson", label="가동률")
ax2.set_ylabel("가동률", color="crimson")

ax1.set_title("차량을 늘리면 승객은 좋아지고 차량은 논다")
plt.tight_layout();

평균 대기시간은 배차된 요청만 대상으로 하므로 실패 요청이 많은 조건에서는 서비스율과 함께 읽습니다.

## 7. 배차 방법별 결과 (교재 11.8)

In [ ]:
rows = []
for n_vehicles in [20, 40, 80]:
    for match in ["greedy", "optimal"]:
        s = reference(demand, vehicles.head(n_vehicles), 1080, 1440, match=match).summary()
        rows.append({
            "차량": n_vehicles,
            "배차": match,
            "평균대기_분": round(s["avg_waiting_time_min"], 2),
            "서비스율": round(s["service_rate"], 3),
        })

pd.DataFrame(rows).pivot(index="차량", columns="배차", values="평균대기_분")

이 예제에서는 차량 80대보다 20대와 40대에서 두 배차 방법의 평균 대기시간 차이가 크게 나타납니다.

## 제출할 것

1. 채운 `labs/ch11_simloop.py`
2. 채점 셀의 출력
3. 구현 중 확인한 오류와 수정 내용 3~5줄
4. 로컬 루프와 녹화본의 입력 및 모형 차이 한 문단

## 정리

- 루프는 1분마다 접수·포기·배차·이동·기록을 순서대로 합니다. 순서가 결과를 바꿉니다
- 차량 대수는 상수가 아닙니다. 근무 시간이 있습니다
- 로컬 루프와 녹화본은 요청 수와 소요시간 모형이 다릅니다
- 12장 실습에서는 이 결과를 보고서에 쓸 표와 그림으로 만듭니다